# Find IBAND


In [1]:
from pathlib import Path
import re

def read_efermi(outcar="OUTCAR"):
    # Read the Fermi energy from OUTCAR.
    text = Path(outcar).read_text(errors="ignore")
    match = re.search(r"E-fermi\s*:\s*([-+]?\d+\.\d+)", text)
    if match is None:
        raise RuntimeError("Could not find E-fermi in OUTCAR.")
    return float(match.group(1))

def read_eigenval(eigenval="EIGENVAL"):
    # Read spin-polarized EIGENVAL.
    # For spin-polarized calculations, each band line should contain:
    # band_index energy_up energy_down occupation_up occupation_down
    lines = Path(eigenval).read_text(errors="ignore").splitlines()
    nelect, nkpts, nbands = lines[5].split()[:3]
    nelect = float(nelect)
    nkpts = int(nkpts)
    nbands = int(nbands)
    kpoints = []
    i = 6
    for k in range(nkpts):
        # Skip blank lines
        while i < len(lines) and not lines[i].strip():
            i += 1
        k_line = lines[i].split()
        k_coord = [float(x) for x in k_line[:3]]
        weight = float(k_line[3])
        i += 1
        bands = []
        for _ in range(nbands):
            parts = lines[i].split()
            band_index = int(parts[0])
            if len(parts) >= 5:
                energy_up = float(parts[1])
                energy_down = float(parts[2])
                occ_up = float(parts[3])
                occ_down = float(parts[4])
            else: raise RuntimeError("This EIGENVAL does not look spin-polarized. "
                    "Check whether ISPIN = 2 was used.")
            bands.append({"band": band_index,
                          "up": energy_up,
                          "down": energy_down,
                          "occ_up": occ_up,
                          "occ_down": occ_down,})
            i += 1
        kpoints.append({"k_index": k + 1,
                        "coord": k_coord,
                        "weight": weight,
                        "bands": bands,})

    return {"nelect": nelect,
            "nkpts": nkpts,
            "nbands": nbands,
            "kpoints": kpoints,}

def find_hos_lus_by_efermi(kpoint, efermi):
    # Find the highest occupied state and lowest unoccupied state
    # at one k-point, using E-fermi as the boundary.
    bands = kpoint["bands"]
    result = {}
    for spin in ["up", "down"]:
        occupied = [b for b in bands if b[spin] <= efermi]
        unoccupied = [b for b in bands if b[spin] > efermi]
        if not occupied:
            raise RuntimeError(f"No occupied bands found for spin-{spin}.")
        if not unoccupied:
            raise RuntimeError(f"No unoccupied bands found for spin-{spin}.")
        hos = max(occupied, key=lambda b: b[spin])
        lus = min(unoccupied, key=lambda b: b[spin])
        result[spin] = {"HOS_band": hos["band"],
                        "HOS_energy": hos[spin],
                        "HOS_delta_E": hos[spin] - efermi,
                        "LUS_band": lus["band"],
                        "LUS_energy": lus[spin],
                        "LUS_delta_E": lus[spin] - efermi,}
    return result


In [2]:
## T-Y midpoint for 41 points per segment
# Path: Gamma-Z, Z-T, T-Y, Y-Gamma
# 41 points per segment gives:
# Gamma-Z: 1-41
# Z-T:     42-82
# T-Y:     83-123
# midpoint of T-Y = 103
KPUSE = 103

efermi = read_efermi("OUTCAR")
data = read_eigenval("EIGENVAL")

target = data["kpoints"][KPUSE - 1]
result = find_hos_lus_by_efermi(target, efermi)

print(f"E-fermi = {efermi:.6f} eV")
print(f"NKPTS   = {data['nkpts']}")
print(f"NBANDS  = {data['nbands']}")
print(f"KPUSE   = {KPUSE}")
print(f"k-point = {target['coord']}")
print()

for spin in ["up", "down"]:
    print(f"Spin-{spin}:")
    print( f"HOS: band {result[spin]['HOS_band']}, "
           f"energy = {result[spin]['HOS_energy']:.6f} eV, "
           f"E - Ef = {result[spin]['HOS_delta_E']:.6f} eV")
    print(f"LUS: band {result[spin]['LUS_band']}, "
          f"energy = {result[spin]['LUS_energy']:.6f} eV, "
          f"E - Ef = {result[spin]['LUS_delta_E']:.6f} eV")
    print()


E-fermi = -3.502500 eV
NKPTS   = 164
NBANDS  = 42
KPUSE   = 103
k-point = [0.5, 0.25, 0.0]

Spin-up:
HOS: band 21, energy = -4.895155 eV, E - Ef = -1.392655 eV
LUS: band 22, energy = -3.408788 eV, E - Ef = 0.093712 eV

Spin-down:
HOS: band 21, energy = -4.895158 eV, E - Ef = -1.392658 eV
LUS: band 22, energy = -3.408792 eV, E - Ef = 0.093708 eV

